# Phase 3a — LLM Human-Validation Sample

This notebook labels the blinded 150-headline Phase 3a sample one headline per request. It supports Gemini and Groq, API-key rotation, resumable execution, strict rubric validation, and audit artifacts.

The hidden provenance key is intentionally not read. The default output is a new LLM file; human rater files are never overwritten.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import pandas as pd
from dotenv import load_dotenv
import json_repair
from tqdm.auto import tqdm

ROOT = Path.cwd()
INPUT_PATH = ROOT / 'data' / 'fomc_phase3_validation_rater_1.csv'
OUTPUT_PATH = ROOT / 'data' / 'fomc_phase3_validation_llm.csv'
AUDIT_PATH = ROOT / 'data' / 'fomc_phase3_validation_llm_audit.jsonl'
METADATA_PATH = ROOT / 'data' / 'fomc_phase3_validation_llm_metadata.json'
HUMAN_OUTPUTS = {ROOT / 'data' / 'fomc_phase3_validation_rater_1.csv', ROOT / 'data' / 'fomc_phase3_validation_rater_2.csv'}
EXPECTED_COLUMNS = ['sample_id', 'headline', 'topic_relevance', 'equity_sentiment', 'label_confidence', 'ambiguity_flag', 'reviewer_notes']
LLM = 'GEMINI'  # Change to 'GROQ' for Groq
GEMINI_MODEL_NAME = 'gemini-2.5-flash'
GEMINI_MODEL_NAMES = ['gemini-2.5-flash-lite', 'gemini-2.5-flash']
GROQ_MODEL_NAME = 'openai/gpt-oss-120b'
GROQ_MODEL_NAMES = [GROQ_MODEL_NAME]
CIRCULAR_MODEL = False
MAX_API_RETRIES = 2
API_CALL_DELAY = 4.0
RETRY_BACKOFF_SECONDS = 2.0
OVERWRITE_OUTPUT = False
RUN_LIVE = False  # Set True only in the explicit live-run cell
PROMPT_VERSION = 'phase3-fomc-headline-rubric-v1'

load_dotenv(ROOT / '.env')

def _key_pool(*names: str) -> list[str]:
    for name in names:
        value = os.getenv(name, '')
        keys = [item.strip() for item in value.split(',') if item.strip()]
        if keys:
            return keys
    return []

GOOGLE_API_KEYS = _key_pool('GOOGLE_API_KEYS', 'GOOGLE_API_KEY')
GROQ_API_KEYS = _key_pool('GROQ_API_KEYS', 'GROQ_API_KEY')
assert LLM in {'GEMINI', 'GROQ'}
assert OUTPUT_PATH not in HUMAN_OUTPUTS, 'Refusing to target a human rater file'
print({'provider': LLM, 'model': GEMINI_MODEL_NAME if LLM == 'GEMINI' else GROQ_MODEL_NAME, 'input': str(INPUT_PATH), 'output': str(OUTPUT_PATH), 'configured_keys': len(GOOGLE_API_KEYS if LLM == 'GEMINI' else GROQ_API_KEYS)})

## Frozen rubric prompt

In [ ]:
RUBRIC = '''You are labeling one blinded FOMC validation headline. Use ONLY the supplied headline text. Do not browse, search, infer its date/source/event, use market outcomes, or use any unstated article context.

First assign topic_relevance: R = centrally about an approaching Federal Reserve monetary-policy decision, policy outlook, interest-rate path, or communication bearing on that decision; I = incidental/non-policy subject such as appointments, supervision, payments, administration, or unrelated speech; U = unclear from the headline alone.

Only when relevance is R, assign equity_sentiment for the near-term implication for broad U.S. equities before the FOMC announcement: A = strongly adverse, B = moderately adverse, C = neutral/balanced/mixed/unclear, D = moderately favorable, E = strongly favorable. Use U when there is insufficient information. When relevance is I, equity_sentiment MUST be ''. When relevance is U, use U unless the implication is unambiguous. This is not a hawkish/dovish or social desirability label.

Assign label_confidence: H = follows directly from the headline, M = reasonable alternative exists, L = highly ambiguous or missing necessary context. Assign ambiguity_flag Y when materially conflicting, context-dependent, or able to switch between non-adjacent sentiment classes; otherwise N. Keep reviewer_notes concise and headline-grounded.

Return exactly one JSON object with exactly these keys: topic_relevance, equity_sentiment, label_confidence, ambiguity_flag, reviewer_notes. No markdown and no extra keys.'''
def make_prompt(sample_id: str, headline: str) -> str:
    return f'{RUBRIC}\n\nSAMPLE_ID: {sample_id}\nHEADLINE: {headline}'

def parse_json(text: str) -> dict[str, Any]:
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.I)
    try:
        value = json.loads(cleaned)
    except json.JSONDecodeError:
        value = json_repair.loads(cleaned)
    if not isinstance(value, dict):
        raise ValueError('response is not a JSON object')
    return value

def validate_label(value: dict[str, Any]) -> dict[str, str]:
    required = {'topic_relevance', 'equity_sentiment', 'label_confidence', 'ambiguity_flag', 'reviewer_notes'}
    if set(value) != required:
        raise ValueError(f'exact keys required; got {sorted(value)}')
    relevance, sentiment = value['topic_relevance'], value['equity_sentiment']
    if relevance not in {'R', 'I', 'U'} or sentiment not in {'', 'A', 'B', 'C', 'D', 'E', 'U'}:
        raise ValueError('invalid relevance or sentiment code')
    if relevance == 'I' and sentiment != '':
        raise ValueError('I relevance requires blank sentiment')
    if relevance == 'U' and sentiment not in {'A', 'B', 'C', 'D', 'E', 'U'}:
        raise ValueError('U relevance requires A-E or U sentiment')
    if value['label_confidence'] not in {'H', 'M', 'L'} or value['ambiguity_flag'] not in {'Y', 'N'}:
        raise ValueError('invalid confidence or ambiguity code')
    if not isinstance(value['reviewer_notes'], str):
        raise ValueError('reviewer_notes must be text')
    return {key: str(value[key]) for key in required}

In [ ]:
# API-free tests: run before any live call.
prompt = make_prompt('FOMCVAL-TEST', 'The Fed is expected to hold rates steady.')
assert 'FOMCVAL-TEST' in prompt and 'The Fed is expected' in prompt
for forbidden in ['event_id', 'published_date', 'source_band', 'headline_key', 'fomc_phase3_validation_key.csv']:
    assert forbidden not in prompt.lower()
assert validate_label({'topic_relevance':'R','equity_sentiment':'C','label_confidence':'H','ambiguity_flag':'N','reviewer_notes':'Policy outlook is central.'})['topic_relevance'] == 'R'
try:
    validate_label({'topic_relevance':'I','equity_sentiment':'C','label_confidence':'H','ambiguity_flag':'N','reviewer_notes':''})
    raise AssertionError('invalid I/C label was accepted')
except ValueError:
    pass
assert parse_json('{"topic_relevance":"U","equity_sentiment":"U","label_confidence":"L","ambiguity_flag":"Y","reviewer_notes":"unclear"}')['topic_relevance'] == 'U'
print('Prompt, parsing, and label-validation tests passed.')

## Provider dispatch and API rotation

In [ ]:
class RotationState:
    def __init__(self):
        self.key_index = {'GEMINI': 0, 'GROQ': 0}
        self.model_index = {'GEMINI': 0, 'GROQ': 0}

    def advance(self, provider: str, key_count: int) -> None:
        self.key_index[provider] = (self.key_index[provider] + 1) % key_count
        if provider == 'GEMINI' and CIRCULAR_MODEL and self.key_index[provider] == 0:
            self.model_index[provider] = (self.model_index[provider] + 1) % len(GEMINI_MODEL_NAMES)

rotation = RotationState()

def provider_call(prompt: str, provider: str, state: RotationState) -> tuple[str, int, str]:
    keys = GOOGLE_API_KEYS if provider == 'GEMINI' else GROQ_API_KEYS
    if not keys:
        raise RuntimeError(f'No API key configured for {provider}')
    key_index = state.key_index[provider]
    key = keys[key_index]
    if provider == 'GEMINI':
        import google.generativeai as genai
        model_name = GEMINI_MODEL_NAMES[state.model_index['GEMINI']] if CIRCULAR_MODEL else GEMINI_MODEL_NAME
        genai.configure(api_key=key)
        model = genai.GenerativeModel(model_name, generation_config={'response_mime_type':'application/json', 'temperature':0.0})
        text = model.generate_content(prompt).text.strip()
    else:
        from groq import Groq
        model_name = GROQ_MODEL_NAMES[state.model_index['GROQ']]
        response = Groq(api_key=key).chat.completions.create(messages=[{'role':'user','content':prompt}], temperature=0.0, model=model_name, max_completion_tokens=2048)
        text = response.choices[0].message.content
    return text, key_index, model_name

# Rotation test with two fake keys; no provider is contacted.
old_keys = GOOGLE_API_KEYS
GOOGLE_API_KEYS = ['redacted-key-0', 'redacted-key-1']
test_state = RotationState()
test_state.advance('GEMINI', 2)
assert test_state.key_index['GEMINI'] == 1
test_state.advance('GEMINI', 2)
assert test_state.key_index['GEMINI'] == 0
GOOGLE_API_KEYS = old_keys
print('API rotation tests passed; secrets are never written to logs.')

## Load, label, resume, and write

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def load_input() -> pd.DataFrame:
    frame = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)
    if list(frame.columns) != EXPECTED_COLUMNS or len(frame) != 150:
        raise ValueError(f'expected 150 rows and exact schema, got {len(frame)} rows / {list(frame.columns)}')
    if frame['sample_id'].duplicated().any() or frame['sample_id'].eq('').any() or frame['headline'].eq('').any():
        raise ValueError('sample IDs and headlines must be non-empty and unique')
    return frame

def is_valid_row(row: pd.Series) -> bool:
    try:
        validate_label({key: row[key] for key in ['topic_relevance','equity_sentiment','label_confidence','ambiguity_flag','reviewer_notes']})
        return True
    except (ValueError, TypeError):
        return False

def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', suffix='.csv', dir=path.parent, delete=False, encoding='utf-8', newline='') as handle:
        temp_path = Path(handle.name)
        frame.to_csv(handle, index=False)
    temp_path.replace(path)

def label_sample(request: Callable[[str], tuple[str, int, str]] | None = None) -> pd.DataFrame:
    if OUTPUT_PATH.exists() and not OVERWRITE_OUTPUT:
        existing = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False)
        if len(existing) and not set(existing['sample_id']).issubset(set(load_input()['sample_id'])):
            raise ValueError('existing output contains unknown sample IDs')
    else:
        existing = pd.DataFrame(columns=EXPECTED_COLUMNS)
    frame = load_input()
    prior = existing.set_index('sample_id') if len(existing) else pd.DataFrame(index=pd.Index([], name='sample_id'))
    state = RotationState()
    audit = []
    for index, row in tqdm(frame.iterrows(), total=len(frame), desc=f'Labeling {LLM}', unit='headline'):
        sample_id = row['sample_id']
        if sample_id in prior.index and is_valid_row(prior.loc[sample_id]):
            frame.loc[index, EXPECTED_COLUMNS[2:]] = prior.loc[sample_id, EXPECTED_COLUMNS[2:]].values
            continue
        prompt = make_prompt(sample_id, row['headline'])
        result = None
        for attempt in range(MAX_API_RETRIES + 1):
            started = datetime.now(timezone.utc).isoformat()
            try:
                raw, key_index, model_name = request(prompt) if request else provider_call(prompt, LLM, state)
                parsed = validate_label(parse_json(raw))
                result = parsed
                audit.append({'sample_id':sample_id, 'status':'ok', 'provider':LLM, 'model':model_name, 'key_index':key_index, 'attempt':attempt, 'raw_response':raw, 'started_at':started})
                break
            except Exception as exc:
                key_count = len(GOOGLE_API_KEYS if LLM == 'GEMINI' else GROQ_API_KEYS)
                key_index = state.key_index[LLM] if key_count else None
                safe_error = str(exc)
                for secret in GOOGLE_API_KEYS + GROQ_API_KEYS:
                    safe_error = safe_error.replace(secret, '[REDACTED]')
                audit.append({'sample_id':sample_id, 'status':'error', 'provider':LLM, 'model':GEMINI_MODEL_NAME if LLM == 'GEMINI' else GROQ_MODEL_NAME, 'key_index':key_index, 'attempt':attempt, 'error_type':type(exc).__name__, 'error':safe_error, 'started_at':started})
                if attempt < MAX_API_RETRIES and key_count:
                    state.advance(LLM, key_count)
                    time.sleep(RETRY_BACKOFF_SECONDS * (2 ** attempt))
        if result:
            for key, value in result.items():
                frame.loc[index, key] = value
        if API_CALL_DELAY and request is None:
            time.sleep(API_CALL_DELAY)
    atomic_csv(frame[EXPECTED_COLUMNS], OUTPUT_PATH)
    with AUDIT_PATH.open('a', encoding='utf-8') as handle:
        for item in audit:
            handle.write(json.dumps(item, ensure_ascii=False) + '\n')
    return frame[EXPECTED_COLUMNS]

In [ ]:
def final_validate(frame: pd.DataFrame) -> None:
    if list(frame.columns) != EXPECTED_COLUMNS or len(frame) != 150 or frame['sample_id'].duplicated().any():
        raise AssertionError('output schema, count, or IDs are invalid')
    source = load_input()
    assert frame['sample_id'].tolist() == source['sample_id'].tolist()
    assert frame['headline'].tolist() == source['headline'].tolist()
    invalid = [row['sample_id'] for _, row in frame.iterrows() if not is_valid_row(row)]
    if invalid:
        raise AssertionError(f'unprocessed or invalid rows: {invalid[:10]}')
    assert not set(frame.columns) - set(EXPECTED_COLUMNS)
    print(f'Validated {len(frame)} labels. Output hash: {sha256_file(OUTPUT_PATH)}')

# API-free structural check using a fake response. This writes only when executed deliberately.
fake = {'topic_relevance':'R','equity_sentiment':'C','label_confidence':'H','ambiguity_flag':'N','reviewer_notes':'Headline-only test label.'}
def fake_request(prompt: str):
    return json.dumps(fake), 0, 'fake-model'
print('Set RUN_LIVE = True and execute the next cell for the real run.')

In [ ]:
if RUN_LIVE:
    selected_keys = GOOGLE_API_KEYS if LLM == 'GEMINI' else GROQ_API_KEYS
    if not selected_keys:
        raise RuntimeError(f'Configure {"GOOGLE_API_KEYS" if LLM == "GEMINI" else "GROQ_API_KEYS"} in .env first')
    if OUTPUT_PATH.exists() and not OVERWRITE_OUTPUT:
        print(f'Resuming valid rows in {OUTPUT_PATH}')
    print(f'Running {LLM}; {len(selected_keys)} API key(s); output={OUTPUT_PATH}')
    labeled = label_sample()
    final_validate(labeled)
    metadata = {'prompt_version':PROMPT_VERSION, 'provider':LLM, 'model':GEMINI_MODEL_NAME if LLM == 'GEMINI' else GROQ_MODEL_NAME, 'generated_at':datetime.now(timezone.utc).isoformat(), 'input_path':str(INPUT_PATH), 'input_sha256':sha256_file(INPUT_PATH), 'output_path':str(OUTPUT_PATH), 'output_sha256':sha256_file(OUTPUT_PATH), 'rows':len(labeled), 'api_key_count':len(selected_keys)}
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print(f'Wrote {OUTPUT_PATH} and {METADATA_PATH}')
else:
    print('Dry run only. No API call or output file was written.')